# 00 — Thiết kế thí nghiệm: muốn cải thiện điều gì?

**Câu hỏi:** Có thể nâng chất lượng dự báo nguy cơ ở khoảng 50–75% học kỳ mà vẫn giữ vai trò HCO, VNLS và HCO–VNLS không?

Mục tiêu Accuracy 90–93% là **kỳ vọng**, chưa phải kết quả. Nếu Accuracy tăng nhưng Recall risk giảm mạnh, mô hình bỏ sót sinh viên cần hỗ trợ và không đạt mục tiêu thực tế.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'clht_code').exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
from NghienCuu_HCO_VNLS.code_dung_chung.thuc_nghiem import (
    GOC, NGHIEN_CUU, DATA, KET_QUA, HINH, CAC_MOC,
    tao_thu_muc, duong_dan, chia_nhom, nap_du_lieu, chon_dac_trung,
    tien_xu_ly, bo_sung_ty_le, bo_sung_cua_so_vle,
)
tao_thu_muc()
print('Dữ liệu nguồn:', DATA)
print('Kết quả riêng:', KET_QUA)

Dữ liệu nguồn: CLHT_GB_GITHUB\data
Kết quả riêng: CLHT_GB_GITHUB\NghienCuu_HCO_VNLS\ket_qua


## 1. Quy tắc để so sánh công bằng

1. Cùng `id_student` luôn thuộc cùng một tập qua **mọi cutoff**; tỷ lệ sinh viên xấp xỉ 70/15/15.
2. Tiền xử lý chỉ `fit` trên train. HCO/VNLS chỉ xem train qua GroupKFold; validation dùng để chọn ứng viên; test chỉ đánh giá cuối.
3. Không dùng `final_result`, `risk`, mã sinh viên, sự kiện sau cutoff làm feature. Không dùng điểm nếu không chứng minh được ngày công bố điểm.
4. So sánh cả Accuracy, Precision, Recall, F1 risk, Weighted F1, AP, ROC-AUC, Brier, số feature, số cảnh báo và thời gian.
5. Tại các cutoff khác nhau, tập enrollment đủ điều kiện có thể khác nhau. Đừng gán toàn bộ chênh lệch metric cho riêng thời điểm quan sát.

**Giới hạn quan trọng:** test của các thí nghiệm lịch sử đã từng được xem, nên test hiện tại không còn là holdout hoàn toàn mới đối với cả dự án. Thí nghiệm mới sẽ không tuning trực tiếp trên test; cần thử thêm khóa kỳ học chưa gặp khi có thể.

In [2]:
import pandas as pd
from clht_code.data import source_manifest
from clht_code.evaluation import split_groups
manifest = source_manifest(DATA)
display(pd.DataFrame([{'bảng': k, 'cột': ', '.join(v['columns']), 'dung lượng MB': round(v['bytes'] / 1e6, 2)} for k, v in manifest.items()]))
print('Số bảng nguồn:', len(manifest))

,bảng,cột,dung lượng MB
0,studentInfo.csv,"code_module, code_presentation, id_student, ge...",2.68
1,studentRegistration.csv,"code_module, code_presentation, id_student, da...",0.78
2,assessments.csv,"code_module, code_presentation, id_assessment,...",0.01
3,studentAssessment.csv,"id_assessment, id_student, date_submitted, is_...",5.69
4,studentVle.csv,"code_module, code_presentation, id_student, id...",453.84
5,vle.csv,"id_site, code_module, code_presentation, activ...",0.18
6,courses.csv,"code_module, code_presentation, module_present...",0.00


Số bảng nguồn: 7


## 2. Bảng quyết định trước khi xem điểm

| Chặng | Câu hỏi | Bằng chứng cần thu |
|---|---|---|
| 01–02 | Dữ liệu có hợp lệ theo thời gian? | Số dòng, trùng khóa, điều kiện cutoff, feature mới |
| 03–04 | Feature và model mới có cải thiện? | Cùng split, so sánh validation tại từng cutoff |
| 05 | HCO–VNLS tạo lợi ích nào? | Cùng budget, cùng evaluator, cả HCO/VNLS riêng lẻ |
| 06–07 | Học sâu/ensemble có đáng chi phí? | Đối chứng đúng tập và kiểm tra trade-off |
| 08 | Ứng viên đã khóa có tổng quát? | Test một lần trong nhánh nghiên cứu này; nêu giới hạn lịch sử |

Các bảng `ket_qua/` và hình `hinh/` trong nhánh này là sản phẩm **mới**; dữ liệu và notebook trước đó được giữ nguyên.